# Walk-in dashboard · Sales pipeline (BigQuery → dashboard)

**Purpose.** Replace the Google-Sheet sales feed of the Walk-in Lead Management dashboard with BigQuery.
This notebook extracts raw retail sales, profiles, cleans and enriches them, validates and reconciles the result,
then uploads one **dashboard-ready dataset** to the dashboard's secure admin endpoint.

**How it is used.** The *daily* 08:00 IST sync is done by the Google Apps Script in `apps-script/` (runs as you — no service account).
This notebook is the documented specification of the same cleaning and KPI logic, plus profiling, validation and reconciliation.
Run it on demand in **Google Colab with your own Google login** (no service account needed), or point it at the dashboard's dataset.
DSR (footfall) and Bigin (leads) are untouched; the dashboard combines all three.

| Step | Section |
|---|---|
| 1 | Parameters |
| 2 | Setup, reference data (store master, category rules) |
| 3 | Extract (BigQuery SQL, or a CSV export for testing) |
| 4 | Data profiling |
| 5 | Cleaning & standardization |
| 6 | Feature engineering |
| 7 | Validation checks (critical checks stop the upload) |
| 8 | KPIs |
| 9 | Reconciliation (internal · vs the last upload · vs a reference file) |
| 10 | Dashboard-ready dataset |
| 11 | Upload & run summary |

**Business rules (agreed with the Founders Office, Oct 2026).**
* Gross = quantity × price; **Revenue (headline) = Gross − discount**.
* Cancelled and refunded orders are **kept** (sizes are profiled). The test store `FRIDO_0000` is excluded.
* One order = one Shopify order name (`#MF…`). Dates are IST (`order_date_ist`).

**Privacy.** The repository is public. This notebook never displays or prints phone numbers or names — only counts and totals.
Never commit an executed copy.

## 1 · Parameters
Overridden by papermill in the scheduled run (`SOURCE=bigquery`). For a local test, set `SOURCE='csv'` and `CSV_PATH`.

In [ ]:
# SOURCE: bigquery = query BigQuery with your own login (Colab or gcloud); dashboard = the cleaned dataset already on the dashboard; csv = an export of the query (CSV_PATH)
# START_DATE: first order date to load (the dashboard counts from 1 Sep 2026); END_DATE: empty means yesterday in IST
# DASHBOARD_URL / ADMIN_KEY: only for SOURCE=dashboard or UPLOAD; UPLOAD: True replaces the dashboard's whole dataset (normally the Apps Script does the daily sync)
# REFERENCE_XLSX: optional OLM_Raw.xlsx for a one-off check against the old Sheet; OUTPUT_DIR: local only, never committed
SOURCE = "bigquery"
CSV_PATH = ""
START_DATE = "2026-09-01"
END_DATE = ""
GCP_PROJECT = "frido-429506"
DASHBOARD_URL = ""
UPLOAD = False
REFERENCE_XLSX = ""
OUTPUT_DIR = "out"

## 2 · Setup and reference data

In [ ]:
import os, re, io, json, gzip, datetime as dt
import numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 40)

IST = dt.timezone(dt.timedelta(hours=5, minutes=30))
TODAY_IST = dt.datetime.now(IST).date()
END = dt.date.fromisoformat(END_DATE) if END_DATE else TODAY_IST - dt.timedelta(days=1)
START = dt.date.fromisoformat(START_DATE)
DASHBOARD_URL = (DASHBOARD_URL or os.environ.get("DASHBOARD_URL", "")).rstrip("/")
ADMIN_KEY = os.environ.get("ADMIN_KEY", "")
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHECKS = []   # (severity, name, passed, detail)
def check(name, passed, detail="", severity="critical"):
    CHECKS.append((severity, name, bool(passed), detail))
    print(("PASS " if passed else ("FAIL " if severity == "critical" else "WARN ")) + name + (f" — {detail}" if detail else ""))

print(f"Run date (IST): {TODAY_IST} · loading {START} → {END} · source: {SOURCE}")

In [ ]:
# Store master — mirrors src/stores.js in the dashboard (id → name, city, format)
STORES = pd.DataFrame([
 ("FRIDO_0001","Phoenix Marketcity Vimannagar","Pune","Mall"), ("FRIDO_0002","Amanora Experience Store","Pune","Mall"),
 ("FRIDO_0003","Amar Tech Park","Pune","High street"), ("FRIDO_0005","Elpro PCMC","Pune","Mall"),
 ("FRIDO_0006","Nexus Westend, Aundh","Pune","Mall"), ("FRIDO_0008","Kopa Mall","Pune","Mall"),
 ("FRIDO_0009","Phoenix Mall of Asia","Bangalore","Mall"), ("FRIDO_0010","SkyCity Borivali","Mumbai","Mall"),
 ("FRIDO_0011","Lakeshore Y junction","Hyderabad","Mall"), ("FRIDO_0012","Frido Experience Store Gachibowli","Hyderabad","High street"),
 ("FRIDO_0013","Frido Store Banjara Hills","Hyderabad","High street"), ("FRIDO_0014","Kompally Store","Hyderabad","High street"),
 ("FRIDO_0015","Bhartiya Mall Store","Bangalore","Mall"), ("FRIDO_0016","Lulu Mall","Bangalore","Mall"),
 ("FRIDO_0017","Vegas Mall","Delhi","Mall"), ("FRIDO_0018","Golf Course Road","Gurgaon","High street"),
 ("FRIDO_0019","Omaxe Store","Faridabad","High street"), ("FRIDO_0020","PMC Whitefield","Bangalore","Mall"),
 ("FRIDO_0021","Felix Plaza","Gurgaon","Mall"), ("FRIDO_0022","DLF summit","Gurgaon","High street"),
 ("FRIDO_0023","M5 Bangalore","Bangalore","Mall"), ("FRIDO_0024","Elan Miracle","Gurgaon","Mall"),
 ("FRIDO_0025","Royal Meenakshi","Bangalore","Mall"),
], columns=["store_id","master_name","city","format"])
TEST_STORES = {"FRIDO_0000"}

# Category rules — mirrors src/categories.js (myfrido.com shop categories; first match wins)
CATEGORY_OVERRIDES = {"3D Foot Scan": "Services"}
CATEGORY_RULES = [
 ("Combos", [r"\bcombo\b", r"\bbundle\b", r"\bduo\b", r"\bkit\b"]), ("Maternity", [r"maternity", r"pregnan"]),
 ("Barefoot", [r"barefoot"]), ("Workspace", [r"\bdesk\b", r"monitor arm", r"footrest", r"laptop", r"keyboard", r"\bmouse\b"]),
 ("Chairs", [r"\bchair\b", r"\bstool\b"]), ("Mattress", [r"mattress", r"topper"]), ("Pillows", [r"pillow"]),
 ("Cushions", [r"cushion", r"\bwedge\b", r"backrest", r"coccyx", r"seat\b"]), ("Insoles", [r"insole"]), ("Socks", [r"\bsocks?\b"]),
 ("Footwear", [r"sandal", r"slipper", r"slides?\b", r"shoes?\b", r"sneaker", r"flip", r"clog", r"chappal"]), ("Masks", [r"\bmask\b"]),
 ("Personal Care", [r"therapy", r"nasal", r"massag", r"bath"]),
 ("Orthotics", [r"orthotic", r"orthopaedic", r"orthopedic", r"posture corrector", r"\bknee\b", r"brace", r"heel protector", r"toe separator", r"\bbelt\b", r"wrist", r"elbow", r"\btape\b", r"gloves"]),
]
def category_of(name):
    n = str(name or "").strip()
    if n in CATEGORY_OVERRIDES: return CATEGORY_OVERRIDES[n]
    low = n.lower()
    for cat, pats in CATEGORY_RULES:
        if any(re.search(p, low) for p in pats): return cat
    return "Accessories"

## 3 · Extract
The query is the one provided by the Founders Office, with a date filter added as query parameters (`@start`, `@end`).

In [ ]:
SQL = """
WITH staff_orders AS (
  SELECT id, name, financial_status, customer_phone, created_at,
         cancelled_at, line_items, discount_codes
  FROM `frido-429506.Frido_Local_Dev.shopify_orders_view`
  QUALIFY ROW_NUMBER() OVER (PARTITION BY name ORDER BY _daton_batch_runtime DESC) = 1
),
retail_orders AS (
  SELECT order_name, store_id, store_name, employee_name, delivery_type
  FROM `frido-429506.Frido_Local_Dev.retail_pos_orders_flat`
  QUALIFY ROW_NUMBER() OVER (PARTITION BY order_name ORDER BY order_date_ist DESC) = 1
)
SELECT
  DATE(CAST(t.created_at AS TIMESTAMP), 'Asia/Kolkata') AS order_date_ist,
  t.name AS order_name, r.store_id, r.store_name, r.employee_name, r.delivery_type,
  RIGHT(REGEXP_REPLACE(t.customer_phone, r'\\D', ''), 10) AS customer_phone,
  t.financial_status, t.cancelled_at IS NOT NULL AS is_cancelled,
  li.sku, li.title AS product_name, li.quantity, CAST(li.price AS NUMERIC) AS price,
  IFNULL((SELECT SUM(CAST(da.amount AS NUMERIC)) FROM UNNEST(li.discount_allocations) da), 0) AS discount_amount,
  (li.quantity * CAST(li.price AS NUMERIC))
    - IFNULL((SELECT SUM(CAST(da.amount AS NUMERIC)) FROM UNNEST(li.discount_allocations) da), 0) AS revenue,
  (SELECT STRING_AGG(dc.code, ', ') FROM UNNEST(t.discount_codes) dc) AS discount_codes
FROM staff_orders t
INNER JOIN retail_orders r ON TRIM(t.name) = TRIM(r.order_name)
LEFT JOIN UNNEST(t.line_items) li
WHERE DATE(CAST(t.created_at AS TIMESTAMP), 'Asia/Kolkata') BETWEEN @start AND @end
"""

if SOURCE == "bigquery":
    # Your own Google login — no service account. In Colab a sign-in prompt appears; elsewhere run `gcloud auth application-default login` once.
    try:
        from google.colab import auth; auth.authenticate_user()
    except ImportError:
        pass
    from google.cloud import bigquery
    client = bigquery.Client(project=GCP_PROJECT)
    job = client.query(SQL, job_config=bigquery.QueryJobConfig(query_parameters=[
        bigquery.ScalarQueryParameter("start", "DATE", START), bigquery.ScalarQueryParameter("end", "DATE", END)]))
    raw = job.result().to_dataframe(create_bqstorage_client=False).astype(str)
    print(f"BigQuery job {job.job_id}: {job.total_bytes_processed/1e6:,.1f} MB processed")
elif SOURCE == "dashboard":
    # The cleaned dataset the Apps Script already sent to the dashboard (admin key) — mapped back to the query's columns for this notebook.
    import requests
    ADMIN_KEY = ADMIN_KEY or input("Dashboard admin key: ")
    r = requests.get(f"{DASHBOARD_URL}/admin/sales.csv", params={"key": ADMIN_KEY}, timeout=300); r.raise_for_status()
    d = pd.read_csv(io.StringIO(r.text.lstrip("\ufeff")), dtype=str, keep_default_na=False)
    raw = pd.DataFrame({"order_date_ist": d.order_date, "order_name": d.order_id, "store_id": d.store_id, "store_name": d.store_name,
        "employee_name": d.employee_name, "delivery_type": d.order_type, "customer_phone": d.phone, "financial_status": d.financial_status,
        "is_cancelled": d.is_cancelled, "sku": d.sku, "product_name": d.product_name, "quantity": d.quantity, "price": d.price,
        "discount_amount": d.discount, "revenue": d.net, "discount_codes": d.discount_codes})
    raw = raw[(raw["order_date_ist"] >= START.isoformat()) & (raw["order_date_ist"] <= END.isoformat())]
elif SOURCE == "csv":
    raw = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)
    raw = raw[(raw["order_date_ist"] >= START.isoformat()) & (raw["order_date_ist"] <= END.isoformat())]
else:
    raise ValueError("SOURCE must be 'bigquery', 'dashboard' or 'csv'")
raw = raw.replace({"None": "", "nan": "", "NaN": "", "<NA>": ""})
print(f"Extracted {len(raw):,} lines · {raw['order_name'].nunique():,} orders · {raw['order_date_ist'].min()} → {raw['order_date_ist'].max()}")

## 4 · Data profiling
Aggregates only. Phone numbers are profiled for validity, never displayed.

In [ ]:
EXPECTED = ["order_date_ist","order_name","store_id","store_name","employee_name","delivery_type","customer_phone",
            "financial_status","is_cancelled","sku","product_name","quantity","price","discount_amount","revenue","discount_codes"]
check("All expected columns present", set(EXPECTED) <= set(raw.columns), f"missing: {sorted(set(EXPECTED) - set(raw.columns))}")
check("Query returned rows", len(raw) > 0, f"{len(raw):,} lines")

prof = pd.DataFrame({
    "blank": [int((raw[c].astype(str).str.strip() == "").sum()) for c in EXPECTED],
    "distinct": [raw[c].nunique() for c in EXPECTED],
}, index=EXPECTED)
prof["example"] = [("(hidden)" if c in ("customer_phone", "employee_name") else str(raw[c].iloc[0])[:40]) for c in EXPECTED]
display(prof)

In [ ]:
daily = raw.groupby("order_date_ist").agg(lines=("order_name", "size"), orders=("order_name", "nunique"), stores=("store_id", "nunique"))
display(daily.T)
display(pd.concat({
  "delivery_type": raw["delivery_type"].value_counts(), "financial_status": raw["financial_status"].value_counts(),
  "is_cancelled": raw["is_cancelled"].value_counts()}).to_frame("lines"))
display(raw.groupby("store_id")["order_name"].nunique().sort_values(ascending=False).to_frame("orders").T)

## 5 · Cleaning & standardization

In [ ]:
df = raw.copy()
for c in ["order_name","store_id","store_name","employee_name","delivery_type","financial_status","sku","product_name","discount_codes"]:
    df[c] = df[c].astype(str).str.strip()
df["store_id"] = df["store_id"].str.upper()
df["delivery_type"] = df["delivery_type"].str.upper()
df["financial_status"] = df["financial_status"].str.lower()
for c in ["quantity","price","discount_amount","revenue"]:              # BigQuery NUMERIC arrives as text, e.g. "0E-9"
    df[c] = pd.to_numeric(df[c], errors="coerce")
bad_num = df[["quantity","price","discount_amount","revenue"]].isna().any(axis=1)
check("All numeric fields readable", bad_num.sum() == 0, f"{int(bad_num.sum())} lines unreadable")
df[["quantity","price","discount_amount","revenue"]] = df[["quantity","price","discount_amount","revenue"]].fillna(0).round(2)

df["order_date"] = pd.to_datetime(df["order_date_ist"], format="%Y-%m-%d", errors="coerce")
check("All order dates readable", df["order_date"].notna().all(), f"{int(df['order_date'].isna().sum())} unreadable")
df["order_date"] = df["order_date"].dt.date

df["is_cancelled"] = df["is_cancelled"].astype(str).str.lower().isin(["true", "1", "yes"])
digits = df["customer_phone"].astype(str).str.replace(r"\D", "", regex=True).str[-10:]
df["phone_valid"] = digits.str.fullmatch(r"[6-9]\d{9}")
df["phone"] = np.where(df["phone_valid"], digits, "")              # invalid phones kept as blank: the order still counts, it just can't match a lead
print(f"Phones: {int(df['phone_valid'].sum()):,} valid · {int((~df['phone_valid']).sum()):,} invalid/blank (set to blank)")

exact_dupes = int(df.duplicated(subset=EXPECTED).sum())
print(f"Exact duplicate lines: {exact_dupes} (kept — an order can contain two identical line items)")

## 6 · Feature engineering

In [ ]:
df["gross"] = (df["quantity"] * df["price"]).round(2)
df["discount"] = df["discount_amount"]
df["net"] = (df["gross"] - df["discount"]).round(2)                 # Revenue (headline)
df["discount_pct"] = np.where(df["gross"] > 0, df["discount"] / df["gross"], 0.0)
df["order_type"] = df["delivery_type"].where(df["delivery_type"].isin(["CASH_AND_CARRY", "ONLINE_ORDER"]), "OTHER")
df["is_test_store"] = df["store_id"].isin(TEST_STORES)
df = df.merge(STORES, on="store_id", how="left")
df["category"] = df["product_name"].map(category_of)
df["weekday"] = pd.to_datetime(df["order_date"]).dt.day_name()
df["month"] = pd.to_datetime(df["order_date"]).dt.strftime("%Y-%m")

orders = df.groupby("order_name").agg(order_date=("order_date","first"), dates=("order_date","nunique"), store_id=("store_id","first"),
    stores=("store_id","nunique"), lines=("order_name","size"), units=("quantity","sum"), gross=("gross","sum"),
    discount=("discount","sum"), net=("net","sum"), cancelled=("is_cancelled","max"), status=("financial_status","first"),
    order_type=("order_type","first"), test=("is_test_store","max"))
print(f"{len(orders):,} orders · avg {orders['lines'].mean():.2f} lines and {orders['units'].mean():.2f} units per order")
display(df.groupby("category")["net"].sum().sort_values(ascending=False).round(0).to_frame("revenue").T)

## 7 · Validation checks
The same checks run in the daily Apps Script; there a critical failure stops the send, emails you, and the dashboard keeps its previous data.

In [ ]:
check("Revenue = qty × price − discount on every line", np.allclose(df["net"], df["revenue"], atol=0.02),
      f"{int((~np.isclose(df['net'], df['revenue'], atol=0.02)).sum())} lines differ")
check("No blank order names", (df["order_name"] != "").all())
check("No negative quantity or price", ((df["quantity"] >= 0) & (df["price"] >= 0)).all())
check("Discount never exceeds gross", (df["discount"] <= df["gross"] + 0.01).all(), f"{int((df['discount'] > df['gross'] + 0.01).sum())} lines", severity="warn")
check("Dates inside the requested range", df["order_date"].between(START, END).all())
check("Yesterday is present", (df["order_date"] == END).any(), f"latest date in data: {df['order_date'].max()}")
unknown = sorted(set(df["store_id"]) - set(STORES["store_id"]) - TEST_STORES)
check("All stores known to the dashboard", not unknown, f"unknown store ids: {unknown} — add them to src/stores.js and this notebook", severity="warn")
check("Each order has one store", (orders["stores"] == 1).all(), f"{int((orders['stores'] > 1).sum())} orders span stores", severity="warn")
check("Each order has one date", (orders["dates"] == 1).all(), f"{int((orders['dates'] > 1).sum())} orders span dates", severity="warn")
check("Valid phone on ≥ 95% of lines", df["phone_valid"].mean() >= 0.95, f"{df['phone_valid'].mean():.1%}", severity="warn")
by_day = orders[~orders["test"]].groupby("order_date").size()
trail = by_day[by_day.index < END].tail(7).mean() if len(by_day) > 1 else np.nan
check("Yesterday's order volume is normal (≥ 40% of trailing 7-day average)",
      not (trail > 0) or by_day.get(END, 0) >= 0.4 * trail, f"{by_day.get(END, 0)} orders vs 7-day avg {trail:.0f}", severity="warn")

## 8 · KPIs
Sales-side KPIs for validation. Walk-in attribution (matching to Bigin leads) and capture/footfall KPIs are computed live in the dashboard, so all filters keep working.

In [ ]:
live = df[~df["is_test_store"]]
lo = orders[~orders["test"]]
kpi = pd.Series({
  "Orders": len(lo), "Lines": len(live), "Units": live["quantity"].sum(),
  "Gross (qty × price)": live["gross"].sum(), "Discount": live["discount"].sum(), "Revenue (gross − discount)": live["net"].sum(),
  "AOV (revenue)": live["net"].sum() / max(len(lo), 1), "Discount % of gross": live["discount"].sum() / max(live["gross"].sum(), 1),
  "Cash & carry share of orders": (lo["order_type"] == "CASH_AND_CARRY").mean(),
  "Cancelled orders (kept)": int(lo["cancelled"].sum()), "Refunded / partially refunded orders (kept)": int(lo["status"].isin(["refunded", "partially_refunded"]).sum()),
})
display(kpi.to_frame("value").style.format("{:,.2f}"))
store_kpi = live.groupby(["store_id","master_name","city","format"]).agg(orders=("order_name","nunique"), units=("quantity","sum"),
    gross=("gross","sum"), discount=("discount","sum"), revenue=("net","sum")).reset_index()
store_kpi["aov"] = store_kpi["revenue"] / store_kpi["orders"]
display(store_kpi.sort_values("revenue", ascending=False).round(0))
day_kpi = live.groupby("order_date").agg(orders=("order_name","nunique"), revenue=("net","sum"), gross=("gross","sum")).round(0)
display(day_kpi.tail(10).T)

## 9 · Reconciliation

In [ ]:
# 9a · internal: line totals = order totals
check("Line totals equal order totals", np.isclose(df["net"].sum(), orders["net"].sum()) and np.isclose(df["gross"].sum(), orders["gross"].sum()))

# 9b · against the dataset currently on the dashboard (catches late edits / back-filled days)
prev = None
if DASHBOARD_URL and ADMIN_KEY:
    import requests
    r = requests.get(f"{DASHBOARD_URL}/admin/sales.csv", params={"key": ADMIN_KEY}, timeout=120)
    if r.ok and r.text.strip() and "order_id" in r.text[:500]:
        prev = pd.read_csv(io.StringIO(r.text.lstrip("\ufeff")), dtype=str, keep_default_na=False)
if prev is not None and len(prev):
    prev["net"] = pd.to_numeric(prev["net"]); cur = live.assign(order_date=live["order_date"].astype(str))
    p = prev.groupby("order_date").agg(prev_orders=("order_id","nunique"), prev_revenue=("net","sum"))
    c = cur.groupby("order_date").agg(orders=("order_name","nunique"), revenue=("net","sum"))
    rec = c.join(p, how="outer").fillna(0); rec["Δ revenue"] = (rec["revenue"] - rec["prev_revenue"]).round(0)
    changed = rec[(rec["Δ revenue"].abs() > 1) & (rec.index < str(END))]
    print(f"Days already on the dashboard whose totals changed since the last upload: {len(changed)}")
    display(changed.round(0))
    check("Earlier days stable vs last upload (≤ 1% revenue change)", (changed["Δ revenue"].abs() <= 0.01 * changed["prev_revenue"].clip(lower=1)).all(), f"{len(changed)} days changed", severity="warn")
else:
    print("No previous upload available — skipped (first run, or DASHBOARD_URL/ADMIN_KEY not set).")

In [ ]:
# 9c · optional: against a reference export of the old Google Sheet (e.g. OLM_Raw.xlsx) — one-off migration check
if REFERENCE_XLSX and os.path.exists(REFERENCE_XLSX):
    ref = pd.read_excel(REFERENCE_XLSX, sheet_name="Sales", dtype=str)
    ref["d"] = pd.to_datetime(ref["POS Created At"]).dt.date
    lo_d, hi_d = max(START, ref["d"].min()), min(END, ref["d"].max())
    R = ref[ref["d"].between(lo_d, hi_d)].assign(amt=lambda x: pd.to_numeric(x["Amount Paid"]))
    B = orders[orders.index.isin(df.loc[df["order_date"].between(lo_d, hi_d), "order_name"])]
    ro = R.groupby("Shopify Order ID").agg(amt=("amt","sum"), retexc=("Ret/Exc", lambda s: ",".join(sorted(set(s)))))
    both = ro.index.intersection(B.index)
    gap = (ro.loc[both, "amt"] - B.loc[both, "net"]).round(0); g = ro.loc[both][gap.abs() > 1]
    summary = pd.Series({
      "Period": f"{lo_d} → {hi_d}", "Sheet orders": len(ro), "BigQuery orders": len(B), "Matched": len(both),
      "Only in Sheet": len(ro.index.difference(B.index)), "Only in BigQuery": len(B.index.difference(ro.index)),
      "Only in BigQuery — revenue": round(B.loc[B.index.difference(ro.index), "net"].sum()),
      "Matched orders with a different amount": len(g), "…of which returns/exchanges/cancellations in the Sheet": int(g["retexc"].str.contains("RETURN|CANCEL").sum()),
    })
    display(summary.to_frame("value"))
    only_b = B.loc[B.index.difference(ro.index)]
    display(only_b.groupby("order_date").size().to_frame("BigQuery-only orders").T)
else:
    print("No reference file given — skipped.")

## 10 · Dashboard-ready dataset
One row per order line. Column contract read by the dashboard (`src/clean.js`).

In [ ]:
OUT_COLS = ["order_date","order_id","store_id","store_name","employee_name","order_type","phone","financial_status","is_cancelled",
            "sku","product_name","category","quantity","price","gross","discount","net","discount_codes"]
final = live.rename(columns={"order_name": "order_id"}).assign(order_date=lambda x: x["order_date"].astype(str))[OUT_COLS].copy()
final = final.sort_values(["order_date","store_id","order_id"]).reset_index(drop=True)
check("Final dataset not empty", len(final) > 0, f"{len(final):,} lines")
print(f"Dashboard-ready dataset: {len(final):,} lines · {final['order_id'].nunique():,} orders · {final['store_id'].nunique()} stores · "
      f"{final['order_date'].min()} → {final['order_date'].max()} · revenue ₹{final['net'].sum():,.0f} · test-store lines removed: {int(df['is_test_store'].sum())}")
final.to_csv(os.path.join(OUTPUT_DIR, "sales_clean.csv"), index=False)   # local only

## 11 · Upload & run summary

In [ ]:
failed = [c for c in CHECKS if c[0] == "critical" and not c[2]]
warned = [c for c in CHECKS if c[0] == "warn" and not c[2]]
summary = {
  "run_at_ist": dt.datetime.now(IST).isoformat(timespec="minutes"), "source": SOURCE, "date_min": str(final["order_date"].min()), "date_max": str(final["order_date"].max()),
  "lines": int(len(final)), "orders": int(final["order_id"].nunique()), "revenue": float(round(final["net"].sum(), 2)), "gross": float(round(final["gross"].sum(), 2)),
  "cancelled_orders": int(lo["cancelled"].sum()), "refunded_orders": int(lo["status"].isin(["refunded", "partially_refunded"]).sum()),
  "invalid_phone_lines": int((~live["phone_valid"]).sum()), "exact_duplicate_lines": exact_dupes,
  "checks_passed": sum(c[2] for c in CHECKS), "checks_failed": len(failed), "checks_warned": len(warned),
  "warnings": [f"{c[1]} ({c[3]})" for c in warned],
}
print(json.dumps(summary, indent=1, ensure_ascii=False))
if failed:
    raise SystemExit("Critical checks failed — NOT uploading: " + "; ".join(c[1] for c in failed))

if UPLOAD:
    import requests
    assert DASHBOARD_URL and ADMIN_KEY, "Set DASHBOARD_URL and the ADMIN_KEY environment variable to upload"
    payload = {"columns": OUT_COLS, "rows": json.loads(final.to_json(orient="values")), "summary": summary}
    body = gzip.compress(json.dumps(payload, separators=(",", ":")).encode("utf-8"))
    r = requests.post(f"{DASHBOARD_URL}/admin/upload-sales", params={"key": ADMIN_KEY}, data=body,
                      headers={"content-type": "application/octet-stream"}, timeout=120)
    res = r.json() if r.headers.get("content-type", "").startswith("application/json") else {"ok": False, "error": r.text[:200]}
    if not r.ok or not res.get("ok"):
        raise SystemExit(f"Upload failed ({r.status_code}): {res.get('error')}")
    print(f"Uploaded {res['rows']:,} lines ({res['bytes']/1024:,.0f} KB compressed) at {res['uploadedAt']}")
else:
    print("UPLOAD = False — dry run, nothing sent.")